In [99]:
print("Initialisation du notebook V5")

Initialisation du notebook V5


# Train Kaggle - Model V7

reload

## Pull GIT

In [100]:
GIT=False

In [101]:
if GIT:
        import subprocess
        
        subprocess.run(
            ["git", "pull", "origin", "main"],
            cwd="/kaggle/working/Train_Kaggle_plank_Detector",
            check=True,
        )
        
        print("✅ GitHub mis à jour")

In [102]:
if GIT:
        from pathlib import Path
        import shutil
        
        REPO = Path(
            "/kaggle/working/Train_Kaggle_plank_Detector"
        )
        
        PROJECT = Path(
            "/kaggle/working/PlankEyev2_multipieces"
        )
        
        shutil.copy2(
            REPO / "Plankeye/model/model_v7.py",
            PROJECT / "model_v7.py",
        )
        
        shutil.copy2(
            REPO / "Plankeye/train/train_v7.py",
            PROJECT / "train_v7.py",
        )
        
        shutil.copy2(
            REPO / "Plankeye/train/checkpoint_sync.py",
            PROJECT / "checkpoint_sync.py",
        )
        print("✅ checkpoint_sync.py actualisé")
        print("✅ model_v7.py actualisé")
        print("✅ train_v7.py actualisé")

Cellule 1 — Imports et chemins

In [103]:
from pathlib import Path
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import time

import torch


# ============================================================
# CHEMINS
# ============================================================

WORKING = Path("/kaggle/working")

PROJECT = WORKING / "PlankEyev2_multipieces"
REPO = WORKING / "Train_Kaggle_plank_Detector"

RUN_DIR = PROJECT / "runs" / "plankeye_v7"

PROJECT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# GITHUB
# ============================================================

GITHUB_REPO = (
    "https://github.com/"
    "maaxxe/Train_Kaggle_plank_Detector.git"
)


# ============================================================
# DATASET D'ENTRAINEMENT
# ============================================================

DATA_DIR = Path(
    "/kaggle/input/datasets/max778/plankeye/"
    "data_kaggle_2_propre/data_kaggle_2_propre"
)


# ============================================================
# DATASET DE CHECKPOINTS V7
# ============================================================

KAGGLE_DATASET = "max778/chekpoints-backbone50"

CHECKPOINT_INPUT_CANDIDATES = [
    Path("/kaggle/input/datasets/max778/chekpoints-backbone50"),
    Path("/kaggle/input/chekpoints-backbone50"),
]


# ============================================================
# FICHIERS LOCAUX
# ============================================================

LOCAL_MODEL = PROJECT / "model_v7.py"
LOCAL_TRAIN = PROJECT / "train_v7.py"
LOCAL_SYNC = PROJECT / "checkpoint_sync.py"


# ============================================================
# GPU / DDP
# ============================================================

NUM_GPUS = 2


# ============================================================
# TRAINING
# ============================================================

EPOCHS = 180

# Batch PAR GPU.
BATCH_SIZE = 1

# 1 x 2 GPU x 8 = batch effectif 16.
GRAD_ACCUM = 8

WORKERS = 2

IMAGE_SIZE = 512
VAL_RATIO = 0.10


# ============================================================
# OPTIMIZER
# ============================================================

LEARNING_RATE = 2e-4
BACKBONE_LR_MULT = 0.25
WEIGHT_DECAY = 1e-4


# ============================================================
# PROGRESSIVE UNFREEZE
# ============================================================

UNFREEZE_LAYER4_EPOCH = 4
UNFREEZE_LAYER3_EPOCH = 9
UNFREEZE_ALL_EPOCH = 16


# ============================================================
# SCHEDULER
# ============================================================

WARMUP_EPOCHS = 3.0
MIN_LR_RATIO = 0.03


# ============================================================
# GRADIENT
# ============================================================

MAX_GRAD_NORM = 10.0


# ============================================================
# CHECKPOINTS
# ============================================================

SAVE_EVERY = 5
KAGGLE_UPLOAD_EVERY = 5


# ============================================================
# RESUME
# ============================================================

# True = last.pt > epoch_XXX.pt > best.pt.
AUTO_RESUME = True


# ============================================================
# AFFICHAGE
# ============================================================

print("=" * 88)
print("CONFIGURATION")
print("=" * 88)
print("PROJECT             :", PROJECT)
print("RUN_DIR             :", RUN_DIR)
print("DATA                :", DATA_DIR)
print("Checkpoint dataset  :", KAGGLE_DATASET)
print("GPU demandés        :", NUM_GPUS)
print("Batch / GPU         :", BATCH_SIZE)
print("Grad accum          :", GRAD_ACCUM)
print("Batch effectif      :", BATCH_SIZE * NUM_GPUS * GRAD_ACCUM)
print("Epochs max          :", EPOCHS)
print("LR                  :", LEARNING_RATE)
print("Backbone LR mult    :", BACKBONE_LR_MULT)
print("Backup tous les     :", KAGGLE_UPLOAD_EVERY, "epochs")
print("=" * 88)

CONFIGURATION
PROJECT             : /kaggle/working/PlankEyev2_multipieces
RUN_DIR             : /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7
DATA                : /kaggle/input/datasets/max778/plankeye/data_kaggle_2_propre/data_kaggle_2_propre
Checkpoint dataset  : max778/chekpoints-backbone50
GPU demandés        : 2
Batch / GPU         : 1
Grad accum          : 8
Batch effectif      : 16
Epochs max          : 180
LR                  : 0.0002
Backbone LR mult    : 0.25
Backup tous les     : 5 epochs


Cellule 2 — Vérification des GPU

In [104]:
print("=" * 88)
print("GPU")
print("=" * 88)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.version.cuda)
print("GPU     :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"{props.total_memory / 1024**3:.2f} GiB"
    )

if torch.cuda.device_count() < NUM_GPUS:
    raise RuntimeError(
        f"❌ {NUM_GPUS} GPU requis, seulement "
        f"{torch.cuda.device_count()} détecté(s). "
        "Active GPU T4 x2 dans Kaggle."
    )

print("✅ Configuration GPU valide.")

GPU
PyTorch : 2.10.0+cu128
CUDA    : 12.8
GPU     : 2
GPU 0: Tesla T4 | 14.56 GiB
GPU 1: Tesla T4 | 14.56 GiB
✅ Configuration GPU valide.


Cellule 3 — Clone ou mise à jour GitHub

In [105]:
print("=" * 88)
print("GITHUB")
print("=" * 88)

if REPO.exists() and (REPO / ".git").exists():
    print("Repository déjà présent.")
    print("Mise à jour depuis origin/main...")

    subprocess.run(
        ["git", "fetch", "origin"],
        cwd=REPO,
        check=True,
    )

    # Le projet utilise la branche main.
    subprocess.run(
        ["git", "reset", "--hard", "origin/main"],
        cwd=REPO,
        check=True,
    )

else:
    if REPO.exists():
        shutil.rmtree(REPO)

    print("Clone du repository...")
    subprocess.run(
        ["git", "clone", GITHUB_REPO, str(REPO)],
        check=True,
    )

GIT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

GIT_BRANCH = subprocess.check_output(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    cwd=REPO,
    text=True,
).strip()

print()
print("Branch :", GIT_BRANCH)
print("Commit :", GIT_COMMIT)
print("✅ GitHub prêt.")

GITHUB
Repository déjà présent.
Mise à jour depuis origin/main...
HEAD is now at 9c9b557 train v7_4 si best_angles.pt ou best_corners.pt n’existent pas encore, elle ne réutilise plus les anciens minima numériques de l’historique

Branch : main
Commit : 9c9b55759fbc6a9254fedff6df10bfde5d2b0fc1
✅ GitHub prêt.


Cellule 4 — Copier les fichiers V7 dans le projet Kaggle

In [106]:
print("=" * 88)
print("COPIE DES FICHIERS V7")
print("=" * 88)

SOURCE_MODEL = REPO / "Plankeye" / "model" / "model_v7.py"
SOURCE_TRAIN = REPO / "Plankeye" / "train" / "train_v7.py"
SOURCE_SYNC = REPO / "Plankeye" / "train" / "checkpoint_sync.py"

sources = [
    (SOURCE_MODEL, LOCAL_MODEL),
    (SOURCE_TRAIN, LOCAL_TRAIN),
    (SOURCE_SYNC, LOCAL_SYNC),
]

for source, destination in sources:
    if not source.exists():
        raise FileNotFoundError(
            f"❌ Fichier Git manquant :\n{source}"
        )

    shutil.copy2(source, destination)

    print(
        f"✅ {source.name:<24} -> {destination} "
        f"({destination.stat().st_size / 1024:.1f} KB)"
    )

print("✅ Fichiers V7 copiés.")

COPIE DES FICHIERS V7
✅ model_v7.py              -> /kaggle/working/PlankEyev2_multipieces/model_v7.py (48.5 KB)
✅ train_v7.py              -> /kaggle/working/PlankEyev2_multipieces/train_v7.py (93.9 KB)
✅ checkpoint_sync.py       -> /kaggle/working/PlankEyev2_multipieces/checkpoint_sync.py (7.6 KB)
✅ Fichiers V7 copiés.


Cellule 5 — Vérifier la syntaxe Python

In [107]:
import py_compile

print("=" * 88)
print("VERIFICATION PYTHON")
print("=" * 88)

for script in (LOCAL_MODEL, LOCAL_TRAIN, LOCAL_SYNC):
    py_compile.compile(str(script), doraise=True)
    print(f"✅ Syntaxe OK : {script.name}")


# Lire MODEL_VERSION sans construire le réseau.
model_text = LOCAL_MODEL.read_text(encoding="utf-8")

version_match = re.search(
    r'MODEL_VERSION\s*=\s*["\']([^"\']+)["\']',
    model_text,
)

if not version_match:
    raise RuntimeError(
        "❌ MODEL_VERSION introuvable dans model_v7.py"
    )

CODE_MODEL_VERSION = version_match.group(1)

print()
print("MODEL_VERSION code :", CODE_MODEL_VERSION)

if not CODE_MODEL_VERSION.startswith("v7-"):
    raise RuntimeError(
        f"❌ Ce fichier ne semble pas être V7 : {CODE_MODEL_VERSION}"
    )

print("✅ Modèle V7 confirmé.")

VERIFICATION PYTHON
✅ Syntaxe OK : model_v7.py
✅ Syntaxe OK : train_v7.py
✅ Syntaxe OK : checkpoint_sync.py

MODEL_VERSION code : v7-resnet50-fpn-pan-dual-corners-quality
✅ Modèle V7 confirmé.


Cellule 6 — Vérifier le dataset d'entraînement

In [108]:
print("=" * 88)
print("DATASET")
print("=" * 88)

IMAGES_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"❌ Dataset introuvable :\n{DATA_DIR}"
    )

if not IMAGES_DIR.is_dir():
    raise FileNotFoundError(IMAGES_DIR)

if not LABELS_DIR.is_dir():
    raise FileNotFoundError(LABELS_DIR)

image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp",
}

images = sorted(
    p for p in IMAGES_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
)

labels = sorted(LABELS_DIR.glob("*.txt"))

print("Images :", len(images))
print("Labels :", len(labels))
print("Images :", IMAGES_DIR)
print("Labels :", LABELS_DIR)

if len(images) == 0:
    raise RuntimeError("❌ Aucune image.")

if len(labels) == 0:
    raise RuntimeError("❌ Aucun label.")

print("✅ Dataset accessible.")

DATASET
Images : 758
Labels : 759
Images : /kaggle/input/datasets/max778/plankeye/data_kaggle_2_propre/data_kaggle_2_propre/images
Labels : /kaggle/input/datasets/max778/plankeye/data_kaggle_2_propre/data_kaggle_2_propre/labels
✅ Dataset accessible.


Cellule 7 — Restaurer automatiquement last.pt depuis le Dataset Kaggle

In [109]:
print("=" * 88)
print("RESTAURATION CHECKPOINT V7")
print("=" * 88)


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


CHECKPOINT_INPUT = first_existing(CHECKPOINT_INPUT_CANDIDATES)
LOCAL_LAST = RUN_DIR / "last.pt"


if LOCAL_LAST.exists():
    print("✅ last.pt déjà présent localement.")
    print(
        f"Taille : {LOCAL_LAST.stat().st_size / 1024**2:.2f} MB"
    )

elif CHECKPOINT_INPUT is not None:
    print("Dataset checkpoint trouvé :")
    print(CHECKPOINT_INPUT)
    print()

    fixed_names = [
        "last.pt",
        "best.pt",
        "history.json",
        "history.csv",
        "loss_curve.png",
        "dataset_split.json",
        "run_config.json",
    ]

    restored = 0

    for name in fixed_names:
        source = CHECKPOINT_INPUT / name
        destination = RUN_DIR / name

        if source.exists():
            shutil.copy2(source, destination)
            restored += 1
            print(
                f"✅ {name:<24} "
                f"{destination.stat().st_size / 1024**2:>8.2f} MB"
            )

    # Restaurer également les checkpoints epoch_XXX.pt présents
    # à la racine du dataset.
    for source in sorted(CHECKPOINT_INPUT.glob("epoch_*.pt")):
        destination = RUN_DIR / source.name

        if not destination.exists():
            shutil.copy2(source, destination)
            restored += 1
            print(
                f"✅ {source.name:<24} "
                f"{destination.stat().st_size / 1024**2:>8.2f} MB"
            )

    print()
    print("Fichiers restaurés :", restored)

    if (RUN_DIR / "last.pt").exists():
        print("✅ last.pt restauré.")
    else:
        print(
            "⚠️ Pas de last.pt dans le Dataset. "
            "Le notebook pourra utiliser le dernier epoch_XXX.pt "
            "ou best.pt."
        )

else:
    print("ℹ️ Aucun Dataset checkpoint V7 attaché.")
    print("➡️ Si aucun checkpoint local n'existe, training neuf.")

RESTAURATION CHECKPOINT V7
✅ last.pt déjà présent localement.
Taille : 457.06 MB


Cellule 8 — Examiner le checkpoint avant reprise

In [110]:
print("=" * 88)
print("DETECTION CHECKPOINT / BACKBONE")
print("=" * 88)


def find_resume_checkpoint(run_dir: Path):
    # 1. last.pt
    last_pt = run_dir / "last.pt"
    if last_pt.exists():
        return last_pt

    # 2. epoch_XXX.pt le plus récent
    epochs = []

    for path in run_dir.glob("epoch_*.pt"):
        match = re.fullmatch(r"epoch_(\d+)\.pt", path.name)
        if match:
            epochs.append((int(match.group(1)), path))

    if epochs:
        epochs.sort(key=lambda item: item[0], reverse=True)
        return epochs[0][1]

    # 3. best.pt
    best_pt = run_dir / "best.pt"
    if best_pt.exists():
        return best_pt

    return None


RESUME_CHECKPOINT = (
    find_resume_checkpoint(RUN_DIR)
    if AUTO_RESUME
    else None
)

CHECKPOINT_INFO = {}


if RESUME_CHECKPOINT is None:
    TRAIN_MODE = "fresh"

    print("ℹ️ Aucun checkpoint de reprise.")
    print("➡️ Nouveau training V7.")
    print("➡️ ResNet50 ImageNet pretrained activé.")

else:
    TRAIN_MODE = "resume"

    print("✅ CHECKPOINT DETECTE")
    print("Fichier :", RESUME_CHECKPOINT)
    print(
        f"Taille  : "
        f"{RESUME_CHECKPOINT.stat().st_size / 1024**2:.2f} MB"
    )

    checkpoint = torch.load(
        RESUME_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    epoch_human = checkpoint.get("epoch_human")

    if epoch_human is None:
        epoch_zero = checkpoint.get("epoch")
        epoch_human = (
            int(epoch_zero) + 1
            if epoch_zero is not None
            else None
        )

    CHECKPOINT_INFO = {
        "checkpoint_version": checkpoint.get("checkpoint_version"),
        "model_version": checkpoint.get("model_version"),
        "epoch_human": epoch_human,
        "next_epoch": (
            int(epoch_human) + 1
            if epoch_human is not None
            else None
        ),
        "global_optimizer_step": checkpoint.get("global_optimizer_step"),
        "best_val_loss": checkpoint.get("best_val_loss"),
        "train_loss": checkpoint.get("train_loss"),
        "val_loss": checkpoint.get("val_loss"),
        "backbone_stage": checkpoint.get("backbone_stage"),
        "timestamp": checkpoint.get("timestamp"),
        "has_ema": "ema_state_dict" in checkpoint,
        "has_optimizer": "optimizer_state_dict" in checkpoint,
        "has_scheduler": "scheduler_state_dict" in checkpoint,
        "has_scaler": "scaler_state_dict" in checkpoint,
    }

    del checkpoint

    for key, value in CHECKPOINT_INFO.items():
        print(f"{key:<24}: {value}")

    checkpoint_version = CHECKPOINT_INFO.get("model_version")

    if (
        checkpoint_version
        and checkpoint_version != CODE_MODEL_VERSION
    ):
        raise RuntimeError(
            "❌ VERSION MODELE INCOMPATIBLE\n"
            f"Code       : {CODE_MODEL_VERSION}\n"
            f"Checkpoint : {checkpoint_version}"
        )

    print()
    print("➡️ Reprise complète avec --resume :")
    print("   modèle")
    print("   EMA")
    print("   optimizer")
    print("   scheduler")
    print("   AMP scaler")
    print("   historique")
    print("   epoch")


print()
print("TRAIN_MODE :", TRAIN_MODE)
print("=" * 88)

DETECTION CHECKPOINT / BACKBONE
✅ CHECKPOINT DETECTE
Fichier : /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7/last.pt
Taille  : 457.06 MB
checkpoint_version      : 3
model_version           : v7-resnet50-fpn-pan-dual-corners-quality
epoch_human             : 121
next_epoch              : 122
global_optimizer_step   : 5199
best_val_loss           : 0.4634822061971614
train_loss              : 0.3784526799780882
val_loss                : 0.5294235180083074
backbone_stage          : full_backbone
timestamp               : 2026-09-16 15:29:29
has_ema                 : True
has_optimizer           : True
has_scheduler           : True
has_scaler              : True

➡️ Reprise complète avec --resume :
   modèle
   EMA
   optimizer
   scheduler
   AMP scaler
   historique
   epoch

TRAIN_MODE : resume


Cellule 9 — Configuration complète du training

In [111]:
train_command = [
    sys.executable,
    "-m",
    "torch.distributed.run",

    "--standalone",

    "--nproc_per_node",
    str(NUM_GPUS),

    str(LOCAL_TRAIN),

    "--data",
    str(DATA_DIR),

    "--output",
    str(RUN_DIR),

    "--epochs",
    str(EPOCHS),

    "--batch-size",
    str(BATCH_SIZE),

    "--grad-accum",
    str(GRAD_ACCUM),

    "--workers",
    str(WORKERS),

    "--image-size",
    str(IMAGE_SIZE),

    "--val-ratio",
    str(VAL_RATIO),

    "--lr",
    str(LEARNING_RATE),

    "--backbone-lr-mult",
    str(BACKBONE_LR_MULT),

    "--weight-decay",
    str(WEIGHT_DECAY),

    "--unfreeze-layer4-epoch",
    str(UNFREEZE_LAYER4_EPOCH),

    "--unfreeze-layer3-epoch",
    str(UNFREEZE_LAYER3_EPOCH),

    "--unfreeze-all-epoch",
    str(UNFREEZE_ALL_EPOCH),

    "--warmup-epochs",
    str(WARMUP_EPOCHS),

    "--min-lr-ratio",
    str(MIN_LR_RATIO),

    "--max-grad-norm",
    str(MAX_GRAD_NORM),

    "--save-every",
    str(SAVE_EVERY),

    "--kaggle-dataset",
    KAGGLE_DATASET,

    "--kaggle-upload-every",
    str(KAGGLE_UPLOAD_EVERY),
]


if TRAIN_MODE == "resume":
    train_command.extend(
        [
            "--resume",
            str(RESUME_CHECKPOINT),
        ]
    )


# Fresh = ne PAS ajouter --no-pretrained.
# model_v7.py utilisera ResNet50_Weights.DEFAULT.


command_text = " ".join(str(x) for x in train_command)
EFFECTIVE_BATCH = BATCH_SIZE * NUM_GPUS * GRAD_ACCUM


print("=" * 88)
print("TRAIN COMMAND PRETE")
print("=" * 88)
print("Mode               :", TRAIN_MODE)

if TRAIN_MODE == "resume":
    print("Checkpoint         :", RESUME_CHECKPOINT.name)
    print(
        "Epoch trouvé       :",
        CHECKPOINT_INFO.get("epoch_human"),
    )
    print(
        "Reprise epoch      :",
        CHECKPOINT_INFO.get("next_epoch"),
    )
    print(
        "Backbone stage     :",
        CHECKPOINT_INFO.get("backbone_stage"),
    )
else:
    print("Epoch départ       : 1")
    print("Backbone           : ResNet50 ImageNet pretrained")

print("GPU                :", NUM_GPUS)
print("Batch / GPU        :", BATCH_SIZE)
print("Grad accum         :", GRAD_ACCUM)
print("Batch effectif     :", EFFECTIVE_BATCH)
print("Backup Kaggle      : tous les", KAGGLE_UPLOAD_EVERY, "epochs")

print()
print("=" * 88)
print("COMMANDE COMPLETE")
print("=" * 88)
print(command_text)
print("=" * 88)

TRAIN COMMAND PRETE
Mode               : resume
Checkpoint         : last.pt
Epoch trouvé       : 121
Reprise epoch      : 122
Backbone stage     : full_backbone
GPU                : 2
Batch / GPU        : 1
Grad accum         : 8
Batch effectif     : 16
Backup Kaggle      : tous les 5 epochs

COMMANDE COMPLETE
/usr/bin/python3 -m torch.distributed.run --standalone --nproc_per_node 2 /kaggle/working/PlankEyev2_multipieces/train_v7.py --data /kaggle/input/datasets/max778/plankeye/data_kaggle_2_propre/data_kaggle_2_propre --output /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7 --epochs 180 --batch-size 1 --grad-accum 8 --workers 2 --image-size 512 --val-ratio 0.1 --lr 0.0002 --backbone-lr-mult 0.25 --weight-decay 0.0001 --unfreeze-layer4-epoch 4 --unfreeze-layer3-epoch 9 --unfreeze-all-epoch 16 --warmup-epochs 3.0 --min-lr-ratio 0.03 --max-grad-norm 10.0 --save-every 5 --kaggle-dataset max778/chekpoints-backbone50 --kaggle-upload-every 5 --resume /kaggle/working/PlankEyev2_multi

Cellule 10 — Construire la commande torchrun

In [112]:
print("=" * 88)
print("PRE-FLIGHT CHECK")
print("=" * 88)

checks = {
    "model_v7.py": LOCAL_MODEL.exists(),
    "train_v7.py": LOCAL_TRAIN.exists(),
    "checkpoint_sync.py": LOCAL_SYNC.exists(),
    "dataset": DATA_DIR.exists(),
    "images": (DATA_DIR / "images").is_dir(),
    "labels": (DATA_DIR / "labels").is_dir(),
    f"GPU x{NUM_GPUS}": torch.cuda.device_count() >= NUM_GPUS,
    "train_command": isinstance(train_command, list) and len(train_command) > 0,
}

all_ok = True

for name, state in checks.items():
    print(("✅" if state else "❌"), name)
    all_ok &= bool(state)


if "train_v7.py" not in command_text:
    print("❌ La commande ne lance pas train_v7.py")
    all_ok = False

if "checkpoint_sync.py" in command_text:
    print("❌ checkpoint_sync.py est dans la commande de training")
    all_ok = False

if TRAIN_MODE == "resume":
    if "--resume" not in train_command:
        print("❌ Mode resume mais --resume absent")
        all_ok = False
else:
    if "--resume" in train_command:
        print("❌ Fresh training mais --resume présent")
        all_ok = False

    if "--no-pretrained" in train_command:
        print("❌ Fresh training avec --no-pretrained")
        all_ok = False


if EFFECTIVE_BATCH != 16:
    print(
        f"⚠️ Batch effectif actuel = {EFFECTIVE_BATCH}. "
        "La configuration prévue était 16."
    )


if not all_ok:
    raise RuntimeError("❌ Pre-flight check échoué.")

print()
print("✅ Tout est prêt pour PlankEye V7.")

PRE-FLIGHT CHECK
✅ model_v7.py
✅ train_v7.py
✅ checkpoint_sync.py
✅ dataset
✅ images
✅ labels
✅ GPU x2
✅ train_command

✅ Tout est prêt pour PlankEye V7.


Cellule 11 — Vérification finale AVANT entraînement

In [ ]:
# ============================================================
# ENVIRONNEMENT DDP
# ============================================================

train_env = os.environ.copy()

train_env["OMP_NUM_THREADS"] = "2"
train_env["PYTHONUNBUFFERED"] = "1"
train_env["NCCL_SOCKET_IFNAME"] = "lo"
train_env["NCCL_SOCKET_FAMILY"] = "AF_INET"


print("=" * 88)
print("PLANKEYE V7 - TRAINING")
print("=" * 88)
print("Mode :", TRAIN_MODE)
print()
print("Commande :")
print(command_text)
print()
print("Pour arrêter : bouton Stop de Kaggle.")
print("=" * 88)


start_time = time.time()
interrupted = False
return_code = None


process = subprocess.Popen(
    train_command,
    cwd=PROJECT,
    env=train_env,
    start_new_session=True,
)


try:
    return_code = process.wait()

except KeyboardInterrupt:
    interrupted = True

    print()
    print("=" * 88)
    print("🛑 ARRÊT MANUEL DEMANDÉ")
    print("=" * 88)
    print("Arrêt de torchrun + rank0 + rank1...")

    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass

    try:
        return_code = process.wait(timeout=20)

    except subprocess.TimeoutExpired:
        print("⚠️ Arrêt gracieux trop long -> SIGKILL.")

        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass

        return_code = process.wait()


duration = time.time() - start_time

print()
print("=" * 88)
print("FIN DU PROCESSUS")
print("=" * 88)
print(f"Durée       : {duration / 60:.2f} min")
print(f"Code retour : {return_code}")
print(f"Interrompu  : {interrupted}")
print("=" * 88)


# ============================================================
# BACKUP APRES ARRET MANUEL
# ============================================================

if interrupted:
    last_checkpoint = RUN_DIR / "last.pt"

    if last_checkpoint.exists() and LOCAL_SYNC.exists():
        print()
        print("☁️ Backup du dernier last.pt sauvegardé...")

        stop_sync_command = [
            sys.executable,
            str(LOCAL_SYNC),

            "--run-dir",
            str(RUN_DIR),

            "--dataset",
            KAGGLE_DATASET,

            "--message",
            "PlankEye V7 - sauvegarde après arrêt manuel",
        ]

        try:
            sync_result = subprocess.run(
                stop_sync_command,
                cwd=PROJECT,
                env=train_env,
            )

            if sync_result.returncode == 0:
                print("✅ Backup Kaggle terminé.")
            else:
                print(
                    "⚠️ Backup Kaggle échoué, "
                    "mais le checkpoint local est conservé."
                )

        except Exception as exc:
            print("⚠️ Backup Kaggle impossible :", exc)
            print("Checkpoint local :", last_checkpoint)

    else:
        print(
            "⚠️ Aucun last.pt ou checkpoint_sync.py disponible "
            "pour le backup après arrêt."
        )


if (
    not interrupted
    and return_code not in (None, 0)
):
    raise RuntimeError(
        f"❌ Training terminé avec le code {return_code}"
    )

PLANKEYE V7 - TRAINING
Mode : resume

Commande :
/usr/bin/python3 -m torch.distributed.run --standalone --nproc_per_node 2 /kaggle/working/PlankEyev2_multipieces/train_v7.py --data /kaggle/input/datasets/max778/plankeye/data_kaggle_2_propre/data_kaggle_2_propre --output /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7 --epochs 180 --batch-size 1 --grad-accum 8 --workers 2 --image-size 512 --val-ratio 0.1 --lr 0.0002 --backbone-lr-mult 0.25 --weight-decay 0.0001 --unfreeze-layer4-epoch 4 --unfreeze-layer3-epoch 9 --unfreeze-all-epoch 16 --warmup-epochs 3.0 --min-lr-ratio 0.03 --max-grad-norm 10.0 --save-every 5 --kaggle-dataset max778/chekpoints-backbone50 --kaggle-upload-every 5 --resume /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7/last.pt

Pour arrêter : bouton Stop de Kaggle.


[W916 15:52:22.790898885 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


====================Model_V7=====================
====================Train_V7_4=====================
====================Model_V7=====================
====================Train_V7_4=====================


[W916 15:52:26.420578047 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W916 15:52:26.437642494 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
Dataset scan: 100%|██████████| 758/758 [00:27<00:00, 27.71it/s]


Using existing dataset split: /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7/dataset_split.json
Loading checkpoint: /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7/last.pt
Checkpoint selectors | best angle=0.013900 | best corners=0.003814

PLANKEYE V7 TRAINING
Model version          : v7-resnet50-fpn-pan-dual-corners-quality
Data                   : /kaggle/input/datasets/max778/plankeye/data_kaggle_2_propre/data_kaggle_2_propre
Output                 : /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7
Train / Val            : 682 / 76
Objects                : 1825
Dataset signature      : 22612c0ca5dde769
Image size             : 512 x 512
Epochs                 : 180
Batch / GPU            : 1
GPUs / world size      : 2
Gradient accumulation  : 8
Effective batch        : 16
AMP FP16               : True
Channels last          : True
EMA                    : True
Pretrained backbone    : True
Frozen backbone BN     : True
Progressive unfreeze   : True
Unfreeze s

...

Val   129/180: 100%|██████████| 38/38 [00:21<00:00,  1.81it/s, corner=0.0016, hm=0.0025, loss=0.4452]



----------------------------------------------------------------------------------------
EPOCH 129/180  |  8.34 min
Train total=0.365023  Val total=0.582619  Best=0.463482
Train corner_delta=0.002686  abs=0.006337  recon=0.002701  heatmap=0.011933
Val   corner_delta=0.003281  abs=0.006185  recon=0.003301  heatmap=0.201211
Select angle=0.015511 (best=0.013900) | corners=0.003772 (best=0.003320)
LR: backbone_decay=1.081e-05, head_decay=4.323e-05, head_no_decay=4.323e-05
Backbone stage: full_backbone | trainable 23.45M / 23.51M
----------------------------------------------------------------------------------------


Val   130/180: 100%|██████████| 38/38 [00:20<00:00,  1.82it/s, corner=0.0012, hm=0.0010, loss=0.5919]



----------------------------------------------------------------------------------------
EPOCH 130/180  |  8.35 min
Train total=0.356717  Val total=0.515396  Best=0.463482
Train corner_delta=0.002505  abs=0.006576  recon=0.002519  heatmap=0.012981
Val   corner_delta=0.003064  abs=0.005639  recon=0.003083  heatmap=0.146607
Select angle=0.014945 (best=0.013900) | corners=0.003499 (best=0.003320)
LR: backbone_decay=1.047e-05, head_decay=4.188e-05, head_no_decay=4.188e-05
Backbone stage: full_backbone | trainable 23.45M / 23.51M
----------------------------------------------------------------------------------------

☁️  Upload Kaggle lancé en arrière-plan : max778/chekpoints-backbone50 (epoch 130)
    Log : /kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7/kaggle_upload_epoch_130.log


Train 131/180:  45%|████▍     | 152/341 [03:34<04:33,  1.45s/it, corner=0.0020, hm=0.0015, loss=0.3324, lr=4.13e-05]

# enregister last best et graphs

ou alors manuel

In [ ]:
!grep -n "best_angles\|best_corners" \
/kaggle/working/PlankEyev2_multipieces/checkpoint_sync.py

In [ ]:
print("=" * 88)
print("BACKUP MANUEL V7")
print("=" * 88)

last_pt = RUN_DIR / "last.pt"

if not LOCAL_SYNC.exists():
    raise FileNotFoundError(
        f"checkpoint_sync.py introuvable :\n{LOCAL_SYNC}"
    )

if not last_pt.exists():
    raise FileNotFoundError(
        f"last.pt introuvable :\n{last_pt}"
    )

backup_command = [
    sys.executable,
    str(LOCAL_SYNC),

    "--run-dir",
    str(RUN_DIR),

    "--dataset",
    KAGGLE_DATASET,

    "--message",
    "PlankEye V7 - checkpoint manuel",
]

print("Commande :")
print(" ".join(str(x) for x in backup_command))
print()

result = subprocess.run(
    backup_command,
    cwd=PROJECT,
    env=os.environ.copy(),
)

if result.returncode != 0:
    raise RuntimeError(
        f"❌ Upload Kaggle échoué : code {result.returncode}"
    )

print("✅ Backup V7 terminé.")

Cellule 13 — Voir les fichiers sauvegardés

In [ ]:
print("=" * 80)
print("FICHIERS DU RUN")
print("=" * 80)


if not RUN_DIR.exists():

    print(
        "Aucun dossier de run."
    )

else:

    files = sorted(
        p
        for p in RUN_DIR.iterdir()
        if p.is_file()
    )


    for path in files:

        size_mb = (
            path.stat().st_size
            / 1024**2
        )

        print(
            f"{path.name:<40} "
            f"{size_mb:>10.2f} MB"
        )

Cellule 14 — Vérifier précisément last.pt

In [ ]:
import torch


LAST_PT = (
    RUN_DIR
    / "last.pt"
)


print("=" * 80)
print("LAST CHECKPOINT")
print("=" * 80)


if not LAST_PT.exists():

    print(
        "❌ last.pt absent"
    )

else:

    checkpoint = torch.load(
        LAST_PT,
        map_location="cpu",
        weights_only=False,
    )


    fields = [
        "checkpoint_version",
        "model_version",
        "epoch_human",
        "global_optimizer_step",
        "best_val_loss",
        "train_loss",
        "val_loss",
        "backbone_stage",
        "timestamp",
    ]


    for field in fields:

        print(
            f"{field:<24} : "
            f"{checkpoint.get(field, '?')}"
        )


    print()
    print(
        "Learning rates :"
    )

    for name, value in (
        checkpoint.get(
            "learning_rates",
            {},
        ).items()
    ):

        print(
            f"  {name:<24} "
            f"{value:.6e}"
        )


    print()
    print(
        "Progressive unfreeze :"
    )

    print(
        json.dumps(
            checkpoint.get(
                "progressive_unfreeze",
                {},
            ),
            indent=2,
        )
    )


    del checkpoint

# courbes 

In [ ]:
# ============================================================
# PLANKEYE V7 - ANALYSE COMPLETE DE L'APPRENTISSAGE
# Depuis epoch 0, à partir de TOUS les checkpoints .pt
# ============================================================

from pathlib import Path
import gc
import json
import math
import warnings

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tqdm.auto import tqdm


# ============================================================
# CONFIG
# ============================================================

RUN_DIR = Path(
    "/kaggle/working/PlankEyev2_multipieces/runs/plankeye_v7"
)

OUTPUT_DIR = RUN_DIR / "learning_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SMOOTH_WINDOW = 5

# Evite de relire les copies temporaires utilisées pour les uploads Kaggle.
IGNORE_DIR_NAMES = {
    "_kaggle_upload",
    "__pycache__",
}

print("=" * 100)
print("PLANKEYE V7 - ANALYSE COMPLETE APPRENTISSAGE")
print("=" * 100)
print("Run :", RUN_DIR)
print("Out :", OUTPUT_DIR)
print()


# ============================================================
# OUTILS
# ============================================================

def safe_float(value, default=np.nan):
    try:
        value = float(value)
        if math.isfinite(value):
            return value
    except Exception:
        pass
    return default


def flatten_history_item(item):
    """
    Transforme un élément de history en une ligne plate :
        epoch
        train_total
        val_total
        train_corner_delta
        ...
        lr_head_decay
        ...
    """

    row = {}

    epoch = item.get("epoch", None)

    if epoch is None:
        epoch_human = item.get("epoch_human", None)
        if epoch_human is not None:
            epoch = int(epoch_human) - 1

    if epoch is None:
        return None

    row["epoch"] = int(epoch)
    row["epoch_human"] = int(item.get("epoch_human", int(epoch) + 1))

    if "duration_seconds" in item:
        row["duration_seconds"] = safe_float(item["duration_seconds"])

    row["backbone_stage"] = item.get("backbone_stage", "")
    row["backbone_trainable"] = safe_float(
        item.get("backbone_trainable", np.nan)
    )

    for prefix in ("train", "val"):
        metrics = item.get(prefix, {})

        if isinstance(metrics, dict):
            for key, value in metrics.items():
                if isinstance(value, (int, float, np.integer, np.floating)):
                    row[f"{prefix}_{key}"] = safe_float(value)

    learning_rates = item.get("learning_rates", {})

    if isinstance(learning_rates, dict):
        for key, value in learning_rates.items():
            row[f"lr_{key}"] = safe_float(value)

    if "best_val_loss" in item:
        row["best_val_loss_history"] = safe_float(
            item["best_val_loss"]
        )

    return row


def torch_load_safe(path):
    """
    mmap=True réduit fortement le pic RAM avec les gros checkpoints.
    Fallback automatique si la version de PyTorch ne le supporte pas.
    """
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
            mmap=True,
        )
    except (TypeError, RuntimeError):
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )


def rolling(series, window=SMOOTH_WINDOW):
    return series.rolling(
        window=window,
        min_periods=1,
        center=True,
    ).mean()


def available(df, column):
    return (
        column in df.columns
        and df[column].notna().any()
    )


def plot_curve(
    ax,
    df,
    columns,
    title,
    ylabel="Loss",
    smooth=False,
    log=False,
):
    found = False

    for column in columns:
        if not available(df, column):
            continue

        y = df[column]

        if smooth:
            y = rolling(y)

        ax.plot(
            df["epoch"],
            y,
            label=column,
            linewidth=1.7,
        )

        found = True

    ax.set_title(title)
    ax.set_xlabel("Epoch (0-based)")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)

    if log:
        ax.set_yscale("log")

    if found:
        ax.legend(fontsize=8)
    else:
        ax.text(
            0.5,
            0.5,
            "Métrique indisponible",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )


# ============================================================
# TROUVER TOUS LES CHECKPOINTS
# ============================================================

pt_files = []

for path in RUN_DIR.rglob("*.pt"):

    if any(part in IGNORE_DIR_NAMES for part in path.parts):
        continue

    if path.is_file():
        pt_files.append(path)

pt_files = sorted(
    pt_files,
    key=lambda p: (
        p.stat().st_mtime,
        p.name,
    ),
)

print(f"Checkpoints .pt trouvés : {len(pt_files)}")

for p in pt_files:
    print(
        f"  {p.name:<24} "
        f"{p.stat().st_size / 1024**2:8.1f} MB"
    )

print()


# ============================================================
# LECTURE DES CHECKPOINTS
# ============================================================

history_by_epoch = {}
checkpoint_rows = []

errors = []

for pt_path in tqdm(
    pt_files,
    desc="Lecture checkpoints",
    dynamic_ncols=True,
):

    try:
        ckpt = torch_load_safe(pt_path)

        if not isinstance(ckpt, dict):
            raise TypeError(
                f"checkpoint type={type(ckpt).__name__}"
            )

        epoch = ckpt.get("epoch", None)
        epoch_human = ckpt.get("epoch_human", None)

        if epoch is None and epoch_human is not None:
            epoch = int(epoch_human) - 1

        checkpoint_rows.append(
            {
                "file": pt_path.name,
                "path": str(pt_path),
                "epoch": (
                    int(epoch)
                    if epoch is not None
                    else np.nan
                ),
                "epoch_human": (
                    int(epoch_human)
                    if epoch_human is not None
                    else (
                        int(epoch) + 1
                        if epoch is not None
                        else np.nan
                    )
                ),
                "best_val_loss": safe_float(
                    ckpt.get(
                        "best_val_loss",
                        np.nan,
                    )
                ),
                "val_loss": safe_float(
                    ckpt.get(
                        "val_loss",
                        ckpt.get(
                            "val_metrics",
                            {},
                        ).get(
                            "total",
                            np.nan,
                        )
                        if isinstance(
                            ckpt.get("val_metrics", {}),
                            dict,
                        )
                        else np.nan,
                    )
                ),
                "train_loss": safe_float(
                    ckpt.get(
                        "train_loss",
                        ckpt.get(
                            "train_metrics",
                            {},
                        ).get(
                            "total",
                            np.nan,
                        )
                        if isinstance(
                            ckpt.get("train_metrics", {}),
                            dict,
                        )
                        else np.nan,
                    )
                ),
                "model_version": ckpt.get(
                    "model_version",
                    "",
                ),
                "history_length": (
                    len(ckpt.get("history", []))
                    if isinstance(
                        ckpt.get("history", []),
                        list,
                    )
                    else 0
                ),
            }
        )

        # ----------------------------------------------------
        # HISTORIQUE EMBARQUÉ DANS LE CHECKPOINT
        # ----------------------------------------------------

        history = ckpt.get("history", [])

        if isinstance(history, list):

            for item in history:

                if not isinstance(item, dict):
                    continue

                row = flatten_history_item(item)

                if row is None:
                    continue

                ep = int(row["epoch"])

                # Les checkpoints récents contiennent généralement
                # l'historique le plus complet.
                history_by_epoch[ep] = row

        # ----------------------------------------------------
        # METRIQUES DE L'EPOCH DU CHECKPOINT
        # fallback si history absent/incomplet
        # ----------------------------------------------------

        if epoch is not None:

            ep = int(epoch)

            row = history_by_epoch.get(
                ep,
                {
                    "epoch": ep,
                    "epoch_human": ep + 1,
                },
            )

            train_metrics = ckpt.get(
                "train_metrics",
                {},
            )
            val_metrics = ckpt.get(
                "val_metrics",
                {},
            )

            if isinstance(train_metrics, dict):
                for k, v in train_metrics.items():
                    if isinstance(
                        v,
                        (
                            int,
                            float,
                            np.integer,
                            np.floating,
                        ),
                    ):
                        row[f"train_{k}"] = safe_float(v)

            if isinstance(val_metrics, dict):
                for k, v in val_metrics.items():
                    if isinstance(
                        v,
                        (
                            int,
                            float,
                            np.integer,
                            np.floating,
                        ),
                    ):
                        row[f"val_{k}"] = safe_float(v)

            lr_dict = ckpt.get(
                "learning_rates",
                {},
            )

            if isinstance(lr_dict, dict):
                for k, v in lr_dict.items():
                    row[f"lr_{k}"] = safe_float(v)

            history_by_epoch[ep] = row

        del ckpt
        gc.collect()

    except Exception as exc:

        errors.append(
            {
                "file": str(pt_path),
                "error": str(exc),
            }
        )


# ============================================================
# COMPLETER AVEC history.csv
# ============================================================

history_csv = RUN_DIR / "history.csv"

if history_csv.exists():

    try:

        hist_csv_df = pd.read_csv(history_csv)

        for _, src in hist_csv_df.iterrows():

            if "epoch" not in src:
                continue

            ep = int(src["epoch"])

            row = history_by_epoch.get(
                ep,
                {
                    "epoch": ep,
                    "epoch_human": ep + 1,
                },
            )

            for column, value in src.items():

                if pd.isna(value):
                    continue

                # Les données de history.csv sont les plus récentes.
                if column in (
                    "backbone_stage",
                ):
                    row[column] = value
                else:
                    try:
                        row[column] = float(value)
                    except Exception:
                        row[column] = value

            history_by_epoch[ep] = row

        print(
            f"✅ history.csv fusionné : "
            f"{len(hist_csv_df)} lignes"
        )

    except Exception as exc:

        print(
            "⚠️ Lecture history.csv impossible :",
            exc,
        )


# ============================================================
# DATAFRAME FINAL
# ============================================================

if not history_by_epoch:
    raise RuntimeError(
        "Aucun historique d'entraînement trouvé dans les checkpoints."
    )

df = pd.DataFrame(
    [
        history_by_epoch[k]
        for k in sorted(history_by_epoch)
    ]
)

df = df.sort_values("epoch").reset_index(drop=True)

# Force valeurs numériques lorsque possible.
for column in df.columns:

    if column in ("backbone_stage",):
        continue

    try:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )
    except Exception:
        pass


# ============================================================
# METRIQUES DERIVEES
# ============================================================

# Score utilisé par best_corners.pt
if all(
    col in df.columns
    for col in (
        "val_corner_delta",
        "val_corner_abs",
        "val_reconstruction",
    )
):

    df["val_corner_score"] = (
        6.0 * df["val_corner_delta"]
        + 2.0 * df["val_corner_abs"]
        + 4.0 * df["val_reconstruction"]
    ) / 12.0


if all(
    col in df.columns
    for col in (
        "train_corner_delta",
        "train_corner_abs",
        "train_reconstruction",
    )
):

    df["train_corner_score"] = (
        6.0 * df["train_corner_delta"]
        + 2.0 * df["train_corner_abs"]
        + 4.0 * df["train_reconstruction"]
    ) / 12.0


if (
    "train_total" in df.columns
    and "val_total" in df.columns
):

    df["generalization_gap"] = (
        df["val_total"]
        - df["train_total"]
    )


if (
    "duration_seconds" in df.columns
):

    df["duration_minutes"] = (
        df["duration_seconds"] / 60.0
    )


# ============================================================
# SAUVEGARDE CSV
# ============================================================

metrics_csv = OUTPUT_DIR / "learning_metrics_all_epochs.csv"
df.to_csv(
    metrics_csv,
    index=False,
)

checkpoint_df = pd.DataFrame(checkpoint_rows)

if len(checkpoint_df):

    checkpoint_df = checkpoint_df.sort_values(
        ["epoch", "file"],
        na_position="last",
    )

checkpoint_csv = OUTPUT_DIR / "checkpoint_inventory.csv"

checkpoint_df.to_csv(
    checkpoint_csv,
    index=False,
)


# ============================================================
# TROUVER LES MEILLEURS EPOCHS
# ============================================================

best_info = {}


def find_best(column):

    if not available(df, column):
        return None

    subset = df[
        np.isfinite(df[column])
    ]

    if len(subset) == 0:
        return None

    idx = subset[column].idxmin()

    return {
        "epoch": int(df.loc[idx, "epoch"]),
        "epoch_human": int(
            df.loc[idx, "epoch_human"]
        ),
        "value": float(df.loc[idx, column]),
    }


best_info["total"] = find_best(
    "val_total"
)

best_info["angles"] = find_best(
    "val_geom_angle"
)

best_info["corners"] = find_best(
    "val_corner_score"
)


# ============================================================
# RESUME TEXTE
# ============================================================

print()
print("=" * 100)
print("HISTORIQUE RECONSTRUIT")
print("=" * 100)

print(
    f"Epoch minimum : {int(df['epoch'].min())}"
)

print(
    f"Epoch maximum : {int(df['epoch'].max())}"
)

print(
    f"Nombre d'epochs disponibles : {len(df)}"
)

missing_epochs = sorted(
    set(
        range(
            int(df["epoch"].min()),
            int(df["epoch"].max()) + 1,
        )
    )
    - set(df["epoch"].astype(int))
)

if missing_epochs:
    print(
        f"⚠️ Epochs manquants : {missing_epochs}"
    )
else:
    print("✅ Aucun epoch manquant")

print()

if best_info["total"]:
    x = best_info["total"]
    print(
        f"🏆 BEST TOTAL   : epoch {x['epoch']} "
        f"| val_total={x['value']:.6f}"
    )

if best_info["angles"]:
    x = best_info["angles"]
    print(
        f"📐 BEST ANGLES  : epoch {x['epoch']} "
        f"| val_geom_angle={x['value']:.6f}"
    )

if best_info["corners"]:
    x = best_info["corners"]
    print(
        f"📍 BEST CORNERS : epoch {x['epoch']} "
        f"| val_corner_score={x['value']:.6f}"
    )

print()

if errors:

    print(
        f"⚠️ {len(errors)} checkpoint(s) "
        "impossible(s) à lire :"
    )

    for item in errors[:20]:
        print(
            " -",
            item["file"],
            ":",
            item["error"],
        )

else:
    print("✅ Tous les checkpoints ont été lus correctement.")


# ============================================================
# DASHBOARD COMPLET
# ============================================================

fig, axes = plt.subplots(
    4,
    3,
    figsize=(22, 22),
)

axes = axes.flatten()


# ------------------------------------------------------------
# 1. TOTAL LOSS
# ------------------------------------------------------------

ax = axes[0]

plot_curve(
    ax,
    df,
    [
        "train_total",
        "val_total",
    ],
    "1 - Total loss Train / Validation",
)

if available(df, "val_total"):
    ax.plot(
        df["epoch"],
        rolling(df["val_total"]),
        linestyle="--",
        linewidth=2,
        label=f"val_total moyenne mobile {SMOOTH_WINDOW}",
    )
    ax.legend(fontsize=8)


# ------------------------------------------------------------
# 2. TOTAL LOSS SMOOTH
# ------------------------------------------------------------

ax = axes[1]

plot_curve(
    ax,
    df,
    [
        "train_total",
        "val_total",
    ],
    f"2 - Total loss lissée ({SMOOTH_WINDOW} epochs)",
    smooth=True,
)


# ------------------------------------------------------------
# 3. GENERALIZATION GAP
# ------------------------------------------------------------

ax = axes[2]

if available(df, "generalization_gap"):

    ax.plot(
        df["epoch"],
        df["generalization_gap"],
        linewidth=1.5,
        label="val_total - train_total",
    )

    ax.axhline(
        0,
        linewidth=1,
        linestyle="--",
    )

    ax.plot(
        df["epoch"],
        rolling(df["generalization_gap"]),
        linewidth=2,
        label=f"moyenne mobile {SMOOTH_WINDOW}",
    )

    ax.legend(fontsize=8)

else:
    ax.text(
        0.5,
        0.5,
        "Indisponible",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )

ax.set_title("3 - Generalization gap")
ax.set_xlabel("Epoch")
ax.set_ylabel("Val - Train")
ax.grid(True, alpha=0.25)


# ------------------------------------------------------------
# 4. CORNER LOSSES
# ------------------------------------------------------------

plot_curve(
    axes[3],
    df,
    [
        "val_corner_delta",
        "val_corner_abs",
        "val_reconstruction",
    ],
    "4 - Losses coins - Validation",
)


# ------------------------------------------------------------
# 5. CORNER SCORE
# ------------------------------------------------------------

ax = axes[4]

plot_curve(
    ax,
    df,
    [
        "train_corner_score",
        "val_corner_score",
    ],
    "5 - Corner score",
)

if best_info["corners"]:

    b = best_info["corners"]

    ax.scatter(
        [b["epoch"]],
        [b["value"]],
        s=100,
        zorder=10,
        label=f"BEST epoch {b['epoch']}",
    )

    ax.legend(fontsize=8)


# ------------------------------------------------------------
# 6. ANGLE LOSS
# ------------------------------------------------------------

ax = axes[5]

plot_curve(
    ax,
    df,
    [
        "train_geom_angle",
        "val_geom_angle",
    ],
    "6 - Angle loss",
)

if best_info["angles"]:

    b = best_info["angles"]

    ax.scatter(
        [b["epoch"]],
        [b["value"]],
        s=100,
        zorder=10,
        label=f"BEST angle epoch {b['epoch']}",
    )

    ax.legend(fontsize=8)


# ------------------------------------------------------------
# 7. GEOMETRY TOTAL
# ------------------------------------------------------------

plot_curve(
    axes[6],
    df,
    [
        "train_geometry",
        "val_geometry",
    ],
    "7 - Geometry loss",
)


# ------------------------------------------------------------
# 8. GEOMETRY SUB-LOSSES VALIDATION
# ------------------------------------------------------------

plot_curve(
    axes[7],
    df,
    [
        "val_geom_area",
        "val_geom_direction",
        "val_geom_length",
        "val_geom_diagonal",
        "val_geom_angle",
        "val_geom_convexity",
    ],
    "8 - Sous-losses géométriques Validation",
)


# ------------------------------------------------------------
# 9. DETECTION / AUXILIARY HEADS
# ------------------------------------------------------------

plot_curve(
    axes[8],
    df,
    [
        "val_heatmap",
        "val_offset",
        "val_size",
        "val_quality",
    ],
    "9 - Heads de détection - Validation",
)


# ------------------------------------------------------------
# 10. CONSISTENCY
# ------------------------------------------------------------

plot_curve(
    axes[9],
    df,
    [
        "val_dual_consistency",
        "val_center_consistency",
    ],
    "10 - Losses de cohérence",
)


# ------------------------------------------------------------
# 11. LEARNING RATE
# ------------------------------------------------------------

lr_columns = [
    col
    for col in df.columns
    if col.startswith("lr_")
]

plot_curve(
    axes[10],
    df,
    lr_columns,
    "11 - Learning rates",
    ylabel="Learning rate",
    log=True,
)


# ------------------------------------------------------------
# 12. NORMALIZED LEARNING PROGRESS
# ------------------------------------------------------------

ax = axes[11]

progress_columns = [
    "val_total",
    "val_corner_score",
    "val_geom_angle",
]

has_progress = False

for column in progress_columns:

    if not available(df, column):
        continue

    valid = df[column].dropna()

    if len(valid) == 0:
        continue

    initial = float(valid.iloc[0])

    if abs(initial) < 1e-12:
        continue

    normalized = df[column] / initial

    ax.plot(
        df["epoch"],
        normalized,
        linewidth=1.8,
        label=f"{column} / valeur initiale",
    )

    has_progress = True

if has_progress:

    ax.axhline(
        1.0,
        linestyle="--",
        linewidth=1,
    )

    ax.legend(fontsize=8)

else:

    ax.text(
        0.5,
        0.5,
        "Indisponible",
        ha="center",
        va="center",
        transform=ax.transAxes,
    )

ax.set_title("12 - Progression normalisée depuis epoch 0")
ax.set_xlabel("Epoch")
ax.set_ylabel("Ratio vs début")
ax.grid(True, alpha=0.25)


# ============================================================
# MARQUEURS BEST TOTAL
# ============================================================

if best_info["total"]:

    best_epoch = best_info["total"]["epoch"]

    for ax in axes:
        ax.axvline(
            best_epoch,
            linestyle=":",
            linewidth=1,
            alpha=0.5,
        )


fig.suptitle(
    "PlankEye V7 - Analyse complète de l'apprentissage",
    fontsize=20,
)

plt.tight_layout(
    rect=[0, 0, 1, 0.975]
)

dashboard_path = (
    OUTPUT_DIR
    / "dashboard_learning.png"
)

plt.savefig(
    dashboard_path,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# GRAPHE SPECIAL : COMPARAISON DES 3 SELECTEURS
# ============================================================

fig, ax = plt.subplots(
    figsize=(16, 7)
)

selector_columns = [
    "val_total",
    "val_corner_score",
    "val_geom_angle",
]

for column in selector_columns:

    if not available(df, column):
        continue

    values = df[column].astype(float)

    valid = values[
        np.isfinite(values)
    ]

    if len(valid) == 0:
        continue

    # Normalisation par min/max pour comparer les tendances.
    vmin = valid.min()
    vmax = valid.max()

    if abs(vmax - vmin) > 1e-12:
        normalized = (
            values - vmin
        ) / (
            vmax - vmin
        )
    else:
        normalized = values * 0.0

    ax.plot(
        df["epoch"],
        rolling(normalized),
        linewidth=2,
        label=column,
    )

ax.set_title(
    "Comparaison des critères de sélection des checkpoints"
)

ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Score normalisé - plus bas = meilleur"
)

ax.grid(
    True,
    alpha=0.25,
)

ax.legend()

selector_path = (
    OUTPUT_DIR
    / "checkpoint_selectors.png"
)

plt.tight_layout()

plt.savefig(
    selector_path,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# GRAPHE SPECIAL : EVOLUTION RECENTE
# ============================================================

recent_start = max(
    int(df["epoch"].min()),
    int(df["epoch"].max()) - 40,
)

recent = df[
    df["epoch"] >= recent_start
].copy()

fig, ax = plt.subplots(
    figsize=(16, 7)
)

for column in (
    "val_corner_score",
    "val_geom_angle",
):

    if available(recent, column):

        values = recent[column]

        minimum = np.nanmin(values)

        if (
            np.isfinite(minimum)
            and minimum > 0
        ):
            values = values / minimum

        ax.plot(
            recent["epoch"],
            values,
            linewidth=2,
            label=f"{column} relatif au meilleur",
        )

ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
)

ax.set_title(
    f"Raffinement géométrique récent "
    f"(epochs {recent_start} → {int(df['epoch'].max())})"
)

ax.set_xlabel(
    "Epoch"
)

ax.set_ylabel(
    "Ratio par rapport au meilleur"
)

ax.grid(
    True,
    alpha=0.25,
)

ax.legend()

recent_path = (
    OUTPUT_DIR
    / "recent_geometry_progress.png"
)

plt.tight_layout()

plt.savefig(
    recent_path,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# TABLE DES DERNIERS EPOCHS
# ============================================================

display_columns = [
    col
    for col in [
        "epoch",
        "train_total",
        "val_total",
        "val_corner_delta",
        "val_corner_abs",
        "val_reconstruction",
        "val_corner_score",
        "val_geometry",
        "val_geom_angle",
        "val_geom_direction",
        "generalization_gap",
    ]
    if col in df.columns
]

print()
print("=" * 100)
print("20 DERNIERS EPOCHS")
print("=" * 100)

display(
    df[
        display_columns
    ].tail(20)
)


# ============================================================
# TOP 10 CHECKPOINTS / EPOCHS
# ============================================================

if available(df, "val_corner_score"):

    print()
    print("=" * 100)
    print("TOP 10 EPOCHS - CORNERS")
    print("=" * 100)

    display(
        df[
            display_columns
        ]
        .sort_values(
            "val_corner_score"
        )
        .head(10)
    )


if available(df, "val_geom_angle"):

    print()
    print("=" * 100)
    print("TOP 10 EPOCHS - ANGLES")
    print("=" * 100)

    display(
        df[
            display_columns
        ]
        .sort_values(
            "val_geom_angle"
        )
        .head(10)
    )


if available(df, "val_total"):

    print()
    print("=" * 100)
    print("TOP 10 EPOCHS - TOTAL LOSS")
    print("=" * 100)

    display(
        df[
            display_columns
        ]
        .sort_values(
            "val_total"
        )
        .head(10)
    )


# ============================================================
# INVENTAIRE DES CHECKPOINTS
# ============================================================

print()
print("=" * 100)
print("CHECKPOINTS ANALYSES")
print("=" * 100)

if len(checkpoint_df):

    display(
        checkpoint_df[
            [
                c
                for c in [
                    "file",
                    "epoch",
                    "val_loss",
                    "best_val_loss",
                    "history_length",
                ]
                if c in checkpoint_df.columns
            ]
        ]
    )


# ============================================================
# RESUME FINAL JSON
# ============================================================

summary = {
    "run_dir": str(RUN_DIR),
    "epoch_min": int(df["epoch"].min()),
    "epoch_max": int(df["epoch"].max()),
    "epochs_available": int(len(df)),
    "missing_epochs": missing_epochs,
    "checkpoints_scanned": len(pt_files),
    "checkpoint_errors": errors,
    "best": best_info,
    "outputs": {
        "metrics_csv": str(metrics_csv),
        "checkpoint_inventory": str(checkpoint_csv),
        "dashboard": str(dashboard_path),
        "selectors": str(selector_path),
        "recent_geometry": str(recent_path),
    },
}

summary_path = (
    OUTPUT_DIR
    / "learning_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print()
print("=" * 100)
print("TERMINE")
print("=" * 100)

print("📊 Dashboard :")
print(dashboard_path)

print()
print("📈 Comparaison selectors :")
print(selector_path)

print()
print("📐 Progression géométrique récente :")
print(recent_path)

print()
print("📄 Toutes les métriques :")
print(metrics_csv)

print()
print("📦 Inventaire checkpoints :")
print(checkpoint_csv)

print()
print("📋 Résumé JSON :")
print(summary_path)

print("=" * 100)

# contenue du kaggle 

In [ ]:
from pathlib import Path
import os
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

ROOTS = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

IMPORTANT_EXTENSIONS = {
    ".pt", ".pth", ".ckpt",
    ".json", ".csv",
    ".py", ".ipynb",
    ".png", ".jpg", ".jpeg",
    ".txt", ".log",
    ".yaml", ".yml",
}

HIGH_PRIORITY_NAMES = {
    "last.pt",
    "best.pt",
    "best_angles.pt",
    "best_corners.pt",
    "history.csv",
    "history.json",
    "loss_curve.png",
    "dataset_split.json",
    "run_config.json",
    "source_git.json",
    "train_v7.py",
    "model_v7.py",
    "checkpoint_sync.py",
}

IGNORE_DIRS = {
    "__pycache__",
    ".git",
    ".cache",
    ".ipynb_checkpoints",
}

MAX_FILES = 10000


# ============================================================
# HELPERS
# ============================================================

def human_size(size):
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(size)

    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024


def classify(path: Path):
    pstr = str(path)

    if pstr.startswith("/kaggle/input"):
        volatility = "SAFE / INPUT"
    elif pstr.startswith("/kaggle/working"):
        volatility = "VOLATILE / WORKING"
    else:
        volatility = "UNKNOWN"

    name = path.name.lower()
    suffix = path.suffix.lower()

    if path.name in HIGH_PRIORITY_NAMES:
        priority = "CRITIQUE"
    elif suffix in {".pt", ".pth", ".ckpt"}:
        priority = "CRITIQUE"
    elif suffix in {".json", ".csv", ".py", ".ipynb"}:
        priority = "HAUTE"
    elif suffix in {".png", ".jpg", ".jpeg", ".txt", ".log"}:
        priority = "MOYENNE"
    else:
        priority = "FAIBLE"

    return volatility, priority


# ============================================================
# SCAN
# ============================================================

rows = []

for root in ROOTS:
    if not root.exists():
        continue

    count = 0

    for path in root.rglob("*"):

        if count >= MAX_FILES:
            break

        if any(part in IGNORE_DIRS for part in path.parts):
            continue

        if not path.is_file():
            continue

        count += 1

        suffix = path.suffix.lower()

        # garde surtout les fichiers pertinents
        if (
            suffix not in IMPORTANT_EXTENSIONS
            and path.name not in HIGH_PRIORITY_NAMES
        ):
            continue

        try:
            stat = path.stat()
        except OSError:
            continue

        volatility, priority = classify(path)

        rows.append(
            {
                "priorite": priority,
                "volatilite": volatility,
                "taille": human_size(stat.st_size),
                "taille_MB": stat.st_size / 1024**2,
                "nom": path.name,
                "chemin": str(path),
            }
        )


df = pd.DataFrame(rows)

if df.empty:
    print("Aucun fichier pertinent trouvé.")
else:

    priority_order = {
        "CRITIQUE": 0,
        "HAUTE": 1,
        "MOYENNE": 2,
        "FAIBLE": 3,
    }

    volatility_order = {
        "VOLATILE / WORKING": 0,
        "SAFE / INPUT": 1,
        "UNKNOWN": 2,
    }

    df["priority_rank"] = df["priorite"].map(priority_order)
    df["volatile_rank"] = df["volatilite"].map(volatility_order)

    df = df.sort_values(
        ["volatile_rank", "priority_rank", "taille_MB"],
        ascending=[True, True, False],
    )

    df = df.drop(
        columns=["priority_rank", "volatile_rank"]
    )

    print("=" * 100)
    print("FICHIERS PERTINENTS")
    print("=" * 100)

    display(df)

    print()
    print("=" * 100)
    print("FICHIERS VOLATILES A SAUVEGARDER")
    print("=" * 100)

    volatile = df[
        df["volatilite"] == "VOLATILE / WORKING"
    ].copy()

    display(
        volatile[
            [
                "priorite",
                "taille",
                "nom",
                "chemin",
            ]
        ]
    )

    print()
    print("=" * 100)
    print("RESUME")
    print("=" * 100)

    print(
        f"Fichiers pertinents total : {len(df)}"
    )

    print(
        f"Volatiles /working       : {len(volatile)}"
    )

    print(
        f"Safe /input              : "
        f"{(df['volatilite'] == 'SAFE / INPUT').sum()}"
    )

    print(
        f"Volume volatile total    : "
        f"{volatile['taille_MB'].sum():.2f} MB"
    )

    critical = volatile[
        volatile["priorite"] == "CRITIQUE"
    ]

    print(
        f"Fichiers CRITIQUES       : {len(critical)}"
    )

    if len(critical):
        print()
        print("A sauvegarder en priorité :")

        for _, row in critical.iterrows():
            print(
                f"  - {row['nom']:<25} "
                f"{row['taille']:>10} "
                f"{row['chemin']}"
            )